In [ ]:
import numpy as np
from Env import GridWorld
import random

DISCOUNT_FACTOR = 1  # 请根据不同的任务使用具体discount_factor


class Agent:
    def __init__(self, env):
        self.env = env
        self.V = np.zeros(env.nS)

    def next_best_action(self, s, V):
        action_values = np.zeros(self.env.nA)
        for a in range(self.env.nA):
            for prob, next_state, reward, done in self.env.P[s][a]:
                action_values[a] += prob * (reward + DISCOUNT_FACTOR * V[next_state])
        return np.argmax(action_values), np.max(action_values)

    def optimize_random(self, subset_ratio=0.3):
        """
        RandomVI: 随机值迭代算法
        参数:
        - subset_ratio: 每次更新状态的比例（默认0.3）
        """
        THETA = 0.0001
        delta = float("inf")
        round_num = 0
        
        subset_size = max(1, int(self.env.nS * subset_ratio))
        
        while delta > THETA:
            delta = 0
            round_num += 1
            
            # 随机选择要更新的状态子集 Bk
            Bk = random.sample(range(self.env.nS), subset_size)
            
            # 只更新选中的状态
            for s in Bk:
                _, best_action_value = self.next_best_action(s, self.V)
                delta = max(delta, np.abs(best_action_value - self.V[s]))
                self.V[s] = best_action_value
        
        # 提取策略
        policy = np.zeros(self.env.nS)
        for s in range(self.env.nS):
            best_action, _ = self.next_best_action(s, self.V)
            policy[s] = best_action
        
        return policy

# 以下是测试代码
# env = GridWorld()
# agent = Agent(env)
# policy = agent.optimize_random(subset_ratio=0.3)
# print("\nBest Policy")
# print(np.reshape([env.get_action_name(entry) for entry in policy], env.shape))